# 1 · Identify the business needs

To start this project, we first identify the business needs by asking the following question:
What decision does this model support, and who acts on the answer?

# 2· Import all the necessary libraries

The first step of every Jupyter notebook is importing the tools:

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from math import ceil

# 3 · Explore the data

We Upload the train.csv file from the folder directly to this Jupyter notebook to inspect its data properties. We will verify its shape, list its variables and extract some statistical descriptive informations such as central tendency and dispersion.

Before modeling anything, we must take a look at the data presented to us. Exploration tells us what we have, what is broken, what is missing, and which features might carry signal.

In [1]:
train_df=pd.read_csv('train.csv')

print("shape of train.csv DataFrame:",train_df.shape)

NameError: name 'pd' is not defined

In [ ]:
train_df.info()

A CORRIGIR

Description:

Out of all the 39 columns, each representing a variables (or feature), we count 29 variables of the type float, 3 variables of the type integer and 7 variables of the type string. 

Financial Variables:
0- Total a Pagar: Total amount to pay for this specific transaction.
1- Total Bruto: Gross purchase amount of this specific transaction.
2- Total Liquido: Net purchase amount of this specific transaction.
3- Total Imp_: Total of the tax amount associated with this specific transaction.
4- Total Portes: Total of the shipping charges of this specific transaction.
5- Total Desc_ Global: Total of the global discount applied to the transactions.
6- Total Desc_ Linha: Total of the discounts applied to this specific transaction.
7- Custo: Total cost associated with this specific transaction.
8- N_ Detalhes: Number of detail lines or items recorded on the invoice of this specific transaction.

Transaction's History Variables:
9- invoicecountcurrent_purchase: Number of invoices associated with this current specific transaction.

repeatpurchase90d: Indicator of whether the customer makes a repeat purchase within 90 days. Typically, 1 indicates a repeat purchase and 0 indicates no repeat purchase within that period.
dayssinceprevious_purchase: Number of days between the current purchase and the customer's previous purchase.
customertenuredays: Number of days the customer has been associated with the business, based on the available historical data.
priorpurchasecount: Number of purchases made by the customer before the current transaction.
priortotalspend: Total amount spent by the customer on previous purchases.
priormeanpurchase_value: Average value of the customer's previous purchases.
priorstdpurchase_value: Standard deviation of the customer's previous purchase values, measuring spending variability.
priorminpurchase_value: Minimum value among the customer's previous purchases.
priormaxpurchase_value: Maximum value among the customer's previous purchases.
priormeangap_days: Average number of days between the customer's previous purchases.
priorstdgap_days: Standard deviation of the intervals between previous purchases, measuring variability in purchasing frequency.
4. Customer Activity and Spending Over Time
purchasecount30d: Number of purchases made during the previous 30 days.
spend_30d: Total amount spent during the previous 30 days.
purchasecount90d: Number of purchases made during the previous 90 days.
spend_90d: Total amount spent during the previous 90 days.
purchasecount180d: Number of purchases made during the previous 180 days.
spend_180d: Total amount spent during the previous 180 days.
purchasecount365d: Number of purchases made during the previous 365 days.
spend_365d: Total amount spent during the previous 365 days.


Customer and Purchase identification Variables:
29- customer_id: Unique client identifier associated with this specific transaction.
30- payment_type: Payment method used by this specific client on this specific transaction.
31- salesperson: Employee who served a client's specific transaction.
32- commercial_zone: Commercial region where this specific transaction was made.
33- transport_method: Type of transportation used for this specific purchase.
34- warehouse: Warehouse where the item for this specific purchase was stored.
35- document_series: Commercial document series used to classify this specific transaction.
36- purchase_date: Date on which this specific transaction occurred.
37- ID: Identifier of this specific transaction (or DataFrame row).


Additional Variables:
38-random_noise: A variable that appears to contain randomly generated values. Its exact purpose cannot be confirmed from the variable name alone.

The variable repeat_purchase_90d is a binary outcome variable (0 if no purchase was made within 90 days after a specific transaction, or 1 if it was done). Thus, it will become our target. This is not a feature, it is a label.

In [ ]:
train_df.describe().T

In the case of qualitative categorical variables we count, per feature, the quantity of null-values, unique values, the most frequent value and its frequency.

In [ ]:
train_df.describe(include="str").T

The code below creates a table indicating how many and in what proportion NA values are in the columns of our train.csv DataFrame. It is basically used to search the columns that have no missing values.

In [ ]:
rows_with_null = train_df.isna().any(axis=1)
per_column = train_df.isna().sum()

pd.Series({
    "Rows with NA value": f"{rows_with_null.sum():,} of {len(train_df):,}"
                        f" ({100*rows_with_null.mean():.2f}%)",
    "Columns with NA value": f"{(per_column>0).sum()} of {train_df.shape[1]}",
    "Columns without NA values": f"{(per_column==0).sum()} of {train_df.shape[1]}",
    "NA cells": f"{train_df.isna().sum().sum():,}",
    "Column with the most NA values": f"{per_column.idxmax()} ({per_column.max()} nulls)",
}).to_frame("missing values as the file arrives")

In [ ]:
print("Columns with no NA values:")
for col in train_df.columns[~train_df.isna().any()]:
    print(col)

After knowing which 3 columns have no missing values, we need to set the index. Our goal here is to check the variable (feature) column called ID, since one purchase should have its unique ID. Verifying this following question is required: 
Is this ID unique, this is, does it not have repeted values?
If the answers is yes, then we can do it.

In [ ]:
duplicated_purchase_ID=train_df.index.duplicated(keep='first').sum()
print("Number of duplicated values on the purchase ID column in train.csv:",duplicated_purchase_ID)

In [ ]:
train_df.set_index('ID', inplace=True)
print("Index name:", train_df.index.name)

Guaranteed the fact that no repeat_purchase_90d value is null (NA), we proceed to the analysis of the target column. The first step is to count absolute and relative the frequency of 1's and 0's. Our aim is to discover other features patterns inside the distribution of these binary values.

In [ ]:
count_target=train_df['repeat_purchase_90d'].value_counts()
target_prop=count_target.to_frame("count").assign(proportion=lambda d: d["count"]/d["count"].sum())
target_prop

# 4 · Modify the data

In this section we should decide which features seem more relevant; we examine all quantitative features grouped by their target variable, i.e the repeat_purchase_90d value. Here we can check the differences between feature values according to their repeat_purchase_90d value (0 or 1).
One ideia would be discarding all features that display significantly similiar values between repeat_purchase_90d=0 and repeat_purchase_90d=1; a candidate would be the feature variable Total Portes.

In [ ]:
train_df.groupby('repeat_purchase_90d').mean(numeric_only=True).round(3).T

Checking if Total Portes column was removed.

In [ ]:
train_df.drop(['Total Portes'],axis=1,inplace=True)
train_df.head()

Now we will be looking for inconsistencies and errors in this train.csv DataFrame. After identifying rows full of NA values (where customer_id and repeat_purchase_90d feature columns are ignored), we delete them from this DataFrame. We are facing variables MCAR; missing completely at random. In this case, we cannot infer any idea from then and, therefore, they are of no practical use. 
Keeping this type of missing values will only cost memory an may not work well for our future predictions.

In [ ]:
drop_not_null_df=train_df.drop(columns=["customer_id", "repeat_purchase_90d"])
all_missing=drop_not_null_df.isna().all(axis=1)
train_df[all_missing]

#dropping rows filled exclusively with null values
train_df.dropna(how='all')

In [ ]:
# Ploting the distributions of the numerical features
# we want to explore the skewness of each variable
# and to understand if there are negative values where only positive numbers make sense, or extreme values far from the mean
# we also want to see if the scale of some variables is different from others to see if we will need to do scaling

num_cols = train_df.select_dtypes(include=[np.number]).columns
num_cols = [c for c in num_cols if c not in ['ID', 'repeat_purchase_90d']]

train_df[num_cols].hist(figsize=(12, 10), bins=30)
plt.suptitle("Distribution of Numerical Variables")
plt.tight_layout()
plt.show()

Uploading the clientes.csv file from the folder directly to this Jupyter notebook. As usual, we inspect its Data Frame dimension.

In [ ]:
clients_df=pd.read_csv('clientes.csv')

print("shape of clientes.csv DataFrame:"clients_df.shape)

We start by analysing a specific Feature column: customer_id. Again, what we pretend to extract from it are exactly two informations: 
1# How many missing values does this column have?
2# Are there any numbers (i.e, Customer ID) duplicated?
If there is no missing values and no duplicated values in customer_id column, we can use it as an index for listing all the rows, since each of them will have an unique index.

In [ ]:
null_customers=clients_df['customer_id'].isna().sum()
duplicated_customers=clients_df['customer_id'].duplicated(keep='first').sum()
print("Number of non-existent Customer ID in clients.csv:",null_customers)
print("Number of duplicated Customer ID in clients.csv:",duplicated_customers)

Indexing the customer_id column.

In [ ]:
clients_df.set_index('customer_id',drop=True,append=False,inplace=False)